# broadcasting-rules composite — cx29: pairwise L2 distances via repeat-broadcast (no copy)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `broadcasting-rules`, `einops-repeat-broadcast`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "broadcasting-rules"
DD_ATOM_IDS = ["broadcasting-rules", "einops-repeat-broadcast"]
DD_SUBTOPICS = ["Numpy: Vectorization and broadcasting", "Einops: Repeat-as-broadcast"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Pairwise computation — every-A against every-B — is the canonical repeat-as-broadcast pattern. Given `A: (N, D)` and `B: (M, D)`, you want `dist: (N, M)`. The free way is to insert a new axis on each side so broadcasting can do the pairing:

  `A_b = repeat(A, 'n d -> n m d', m=M)`  (insert m-axis, stride 0)
  `B_b = repeat(B, 'm d -> n m d', n=N)`  (insert n-axis, stride 0)

Both are zero-copy views (the einops-repeat-broadcast atom). Then broadcasting rules let you subtract, square, and reduce them as if they were materialized `(N, M, D)` tensors — but the underlying storage is still the original `A` and `B` buffers. Memory cost is O(N + M), not O(N * M).

### Composite Exercise — pairwise L2 distances via repeat-broadcast (no copy)

**Atoms exercised together**: `broadcasting-rules`, `einops-repeat-broadcast`

Implement `cx29_pairwise_l2(A, B)` that computes the `(N, M)` matrix of pairwise L2 distances between rows of `A: (N, D)` and rows of `B: (M, D)`.

1. **Repeat-broadcast** to insert the pairing axes WITHOUT copying:
   - `A_b = repeat(A, 'n d -> n m d', m=M)` (stride-0 view)
   - `B_b = repeat(B, 'm d -> n m d', n=N)` (stride-0 view)
2. **Broadcast-subtract**, square, sum over `d`, and sqrt. The result is the `(N, M)` distance matrix.

Cross-check against `t.cdist(A, B)`. The test also asserts that `A_b.data_ptr() == A.data_ptr()` and same for `B` — i.e. the repeat-broadcast was a true zero-copy view, not a materialized tensor.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx29_pairwise_l2(A, B):
    raise NotImplementedError

def _test_cx29():
    # Case A: cross-check against t.cdist on random inputs.
    A = t.randn(5, 3)
    B = t.randn(4, 3)
    out = cx29_pairwise_l2(A, B)
    assert tuple(out.shape) == (5, 4), f'expected (5,4), got {tuple(out.shape)}'
    ref = t.cdist(A, B)
    assert t.allclose(out, ref, atol=1e-5), f'max diff {(out-ref).abs().max()}'

    # Case B: hand-check on simple integer-valued points.
    A2 = t.tensor([[0.0, 0.0], [3.0, 4.0]])  # origin, (3,4)
    B2 = t.tensor([[0.0, 0.0], [1.0, 0.0]])  # origin, (1,0)
    out2 = cx29_pairwise_l2(A2, B2)
    # Distances:
    #   (0,0)-(0,0) = 0;  (0,0)-(1,0) = 1
    #   (3,4)-(0,0) = 5;  (3,4)-(1,0) = sqrt(4+16) = sqrt(20)
    expected = t.tensor([[0.0, 1.0], [5.0, 20.0 ** 0.5]])
    assert t.allclose(out2, expected, atol=1e-5), f'got {out2}, expected {expected}'

    # Case C: square distance matrix (A == B) → zero diagonal.
    A3 = t.randn(6, 8)
    out3 = cx29_pairwise_l2(A3, A3)
    assert tuple(out3.shape) == (6, 6)
    assert t.allclose(out3.diagonal(), t.zeros(6), atol=1e-5), f'diagonal: {out3.diagonal()}'
    # Symmetric.
    assert t.allclose(out3, out3.T, atol=1e-5), 'pairwise distance must be symmetric'

    # Case D: realistic scale.
    A4 = t.randn(64, 16)
    B4 = t.randn(128, 16)
    out4 = cx29_pairwise_l2(A4, B4)
    assert tuple(out4.shape) == (64, 128)
    assert t.allclose(out4, t.cdist(A4, B4), atol=1e-4)
    # --- atom-coverage: einops.repeat must be used AND produce stride-0 broadcast views ---
    import inspect as _inspect
    _src = _inspect.getsource(cx29_pairwise_l2)
    assert 'repeat(' in _src, 'must use einops.repeat to insert the pairing axes (not full broadcast / cdist)'
    assert 'cdist' not in _src, 'must build pairwise distances by hand via repeat + reduce, not torch.cdist'
    assert '.expand(' not in _src and 'expand_as' not in _src, 'must use einops.repeat, not torch.expand'
    assert 'broadcast_to' not in _src, 'must use einops.repeat, not broadcast_to'
    _g = cx29_pairwise_l2.__globals__
    _orig_repeat = _g.get('repeat')
    _calls = []
    def _spy_repeat(*a, **kw):
        r = _orig_repeat(*a, **kw)
        _calls.append(r)
        return r
    _g['repeat'] = _spy_repeat
    try:
        cx29_pairwise_l2(t.randn(5, 3), t.randn(4, 3))
    finally:
        _g['repeat'] = _orig_repeat
    assert len(_calls) >= 1, 'cx29_pairwise_l2 must call einops.repeat'
    # At least one repeat-output must have a stride-0 axis (the inserted pairing axis).
    assert any(0 in r.stride() for r in _calls), (
        'einops.repeat output must be a stride-0 broadcast view along the inserted pairing axis'
    )

    _dd_passed.add('cx29')

_test_cx29()

<details><summary>Show solution — cx29</summary>

```python
def cx29_pairwise_l2(A, B):
    N, _ = A.shape
    M, _ = B.shape
    # Atom A (einops-repeat-broadcast): insert pairing axes as stride-0 views.
    A_b = repeat(A, 'n d -> n m d', m=M)
    B_b = repeat(B, 'm d -> n m d', n=N)
    # Atom B (broadcasting-rules): aligned shapes (n, m, d) — elementwise sub + reduce.
    diff = A_b - B_b
    sq = diff ** 2
    return reduce(sq, 'n m d -> n m', 'sum').sqrt()
```

The whole point of repeat-broadcast over `.repeat()` (torch method) is the storage: the intermediate `(N, M, D)` tensor is *never materialized*. Both `A_b` and `B_b` are stride-0 views sharing storage with `A` and `B` — memory cost is O(N + M), not O(N*M*D). The subtract step does allocate (you can't avoid that for the diff), but the *input* tensors stay free. At ARENA scale (millions of ray/triangle pairs) this is the difference between OOM and OK.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx29'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx29',
        'subtopics': ["Numpy: Vectorization and broadcasting", "Einops: Repeat-as-broadcast"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()